In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Mounted at /content/drive
True
True


In [2]:
sys.path.append('/content/drive/MyDrive')

In [3]:
"""
t4_t7_posthoc.py  —  T4 (statistical inference) + T7 (per-class metrics)

NO RE-RUN REQUIRED. This script reads the per-seed JSON outputs that are
already in your repository (seed_runs_25ms/, baselines_25ms/, etc.) and
produces:

  T4  per-seed paired differences, paired t-test, Wilcoxon signed-rank,
      bootstrap CI of the mean difference, and paired Cohen's d
      (review item C4 — strengthen statistics for n=20).

  T7  per-class precision / recall / F1 at 15 and 25 m/s, both
      averaged across seeds and on the median-accuracy seed
      (review item M2 — the accuracy vs macro-F1 gap at 15 m/s).

Usage:
    python t4_t7_posthoc.py --results-dir /path/to/repo

The --results-dir must contain: seed_runs_25ms/, seed_runs_15ms/,
baselines_25ms/, baselines_15ms/  (unzip the archives first).

Everything is also written to ./outputs_posthoc/ as JSON + LaTeX.
"""
import os, json, glob, argparse
import numpy as np
from scipy import stats
from common import (compute_metrics, agg, CLASS_NAMES, SEEDS)

In [4]:
# ---- baseline filename prefixes as used in your repo -----------------------
BASELINE_PREFIX = {
    'RF-Stat':       'rf_vrs',
    'BERT WordPiece':'bert_wp',
    'LSTM-RELU-IO-L1':'lstm_relu_iol1',
}

In [5]:
def _load_json(path):
    with open(path) as f:
        return json.load(f)

In [6]:
def load_proposed(results_dir, spd):
    """Per-seed proposed-detector results (full ensemble)."""
    out = {}
    for p in sorted(glob.glob(os.path.join(results_dir, f'seed_runs_{spd}ms',
                                            'seed_*_result.json'))):
        d = _load_json(p)
        out[d['seed']] = d
    return out

In [7]:
def load_baseline(results_dir, spd, prefix):
    out = {}
    for p in sorted(glob.glob(os.path.join(results_dir, f'baselines_{spd}ms',
                                            f'{prefix}_{spd}ms_seed*.json'))):
        d = _load_json(p)
        out[d['seed']] = d
    return out

In [8]:
# ───────────────────────── T4: statistics ──────────────────────────────────
def bootstrap_ci_mean_diff(diffs, n_boot=10000, alpha=0.05, seed=0):
    """Percentile bootstrap CI for the mean of paired differences."""
    rng = np.random.RandomState(seed)
    diffs = np.asarray(diffs, dtype=float)
    boots = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean()
                      for _ in range(n_boot)])
    lo = np.percentile(boots, 100 * alpha / 2)
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return float(lo), float(hi)

In [9]:
def cohens_d_paired(a, b):
    diffs = np.asarray(a) - np.asarray(b)
    sd = diffs.std(ddof=1)
    return float(diffs.mean() / sd) if sd > 0 else 0.0

In [10]:
def paired_comparison(prop, base, name, spd):
    """prop, base: dict seed->result. Returns a stats record (accuracy scale %)."""
    seeds = sorted(set(prop) & set(base))
    a = np.array([prop[s]['accuracy'] for s in seeds]) * 100  # proposed
    b = np.array([base[s]['accuracy'] for s in seeds]) * 100  # baseline
    diffs = a - b                                             # +ve => proposed better

    # paired t-test (guard against zero-variance diffs, e.g. baseline = 100%)
    if np.std(diffs, ddof=1) > 0:
        t_stat, p_t = stats.ttest_rel(a, b)
    else:
        t_stat, p_t = np.nan, np.nan
    # Wilcoxon signed-rank (needs non-zero diffs); fall back to NaN if degenerate
    try:
        nz = diffs[diffs != 0]
        if len(nz) >= 1 and not np.allclose(diffs, 0):
            w_stat, p_w = stats.wilcoxon(a, b, zero_method='wilcox',
                                         alternative='two-sided', mode='auto')
        else:
            w_stat, p_w = np.nan, np.nan
    except ValueError:
        w_stat, p_w = np.nan, np.nan
    ci_lo, ci_hi = (bootstrap_ci_mean_diff(diffs) if np.std(diffs) > 0
                    else (float(diffs.mean()), float(diffs.mean())))
    return {
        'comparison': f'Proposed vs {name}', 'speed': spd, 'seeds': seeds,
        'proposed_per_seed': a.tolist(), 'baseline_per_seed': b.tolist(),
        'per_seed_diff_pp': diffs.tolist(),
        'mean_diff_pp': float(diffs.mean()),
        'paired_t_p': float(p_t) if not np.isnan(p_t) else None,
        'wilcoxon_p':  float(p_w) if not np.isnan(p_w) else None,
        'bootstrap_ci95_pp': [round(ci_lo, 3), round(ci_hi, 3)],
        'cohens_d_paired': round(cohens_d_paired(a, b), 3),
        'all_same_direction': bool(np.all(diffs > 0) or np.all(diffs < 0)),
    }

In [11]:
def run_T4(results_dir, outdir):
    records = []
    for spd in (25, 15):
        prop = load_proposed(results_dir, spd)
        if not prop:
            print(f'[T4] WARNING: no proposed results for {spd} m/s, skipping')
            continue
        for name, prefix in BASELINE_PREFIX.items():
            base = load_baseline(results_dir, spd, prefix)
            if not base:
                print(f'[T4] WARNING: baseline {name} missing at {spd} m/s')
                continue
            records.append(paired_comparison(prop, base, name, spd))

    # console table
    print('\n' + '=' * 96)
    print('  T4 — Strengthened statistics (n=20): paired t, Wilcoxon, bootstrap CI, paired d')
    print('=' * 96)
    hdr = (f'{"Comparison":<26s}{"Spd":>4s}{"Δmean":>8s}{"t-p":>9s}'
           f'{"Wilcox-p":>10s}{"bootCI95 (pp)":>20s}{"d":>7s}{"same dir":>9s}')
    print(hdr); print('-' * 96)
    for r in records:
        t = f'{r["paired_t_p"]:.4f}' if r['paired_t_p'] is not None else 'n/a'
        w = f'{r["wilcoxon_p"]:.4f}' if r['wilcoxon_p'] is not None else 'n/a'
        ci = f'[{r["bootstrap_ci95_pp"][0]:+.2f}, {r["bootstrap_ci95_pp"][1]:+.2f}]'
        print(f'{r["comparison"]:<26s}{r["speed"]:>4d}{r["mean_diff_pp"]:>+8.2f}'
              f'{t:>9s}{w:>10s}{ci:>20s}{r["cohens_d_paired"]:>+7.2f}'
              f'{("yes" if r["all_same_direction"] else "no"):>9s}')

    with open(os.path.join(outdir, 't4_statistics.json'), 'w') as f:
        json.dump(records, f, indent=2)

    # LaTeX
    lines = [r'\begin{table}[H]',
             r'\caption{Strengthened paired statistics over 20 seeds '
             r'(positive $\Delta$ favours the proposed detector).}',
             r'\centering', r'\begin{tabular}{llrrrrr}', r'\toprule',
             r'Comparison & Regime & $\Delta$ (pp) & $p_{t}$ & $p_{\text{Wilcoxon}}$ '
             r'& Bootstrap 95\% CI (pp) & $d_{\text{paired}}$ \\', r'\midrule']
    for r in records:
        t = f'{r["paired_t_p"]:.4f}' if r['paired_t_p'] is not None else '--'
        w = f'{r["wilcoxon_p"]:.4f}' if r['wilcoxon_p'] is not None else '--'
        ci = f'[{r["bootstrap_ci95_pp"][0]:+.2f}, {r["bootstrap_ci95_pp"][1]:+.2f}]'
        nm = r['comparison'].replace('Proposed vs ', '')
        lines.append(f'{nm} & {r["speed"]} m/s & {r["mean_diff_pp"]:+.2f} & {t} & '
                     f'{w} & {ci} & {r["cohens_d_paired"]:+.2f} \\\\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    with open(os.path.join(outdir, 't4_table.tex'), 'w') as f:
        f.write('\n'.join(lines))
    return records

In [12]:
# ───────────────────────── T7: per-class metrics ───────────────────────────
def run_T7(results_dir, outdir):
    all_rows = []
    for spd in (25, 15):
        prop = load_proposed(results_dir, spd)
        if not prop:
            continue
        seeds = sorted(prop)
        # per-class arrays across seeds (recompute from y_true/y_pred to be safe)
        P = {c: [] for c in range(3)}
        R = {c: [] for c in range(3)}
        F = {c: [] for c in range(3)}
        accs = []
        # NEW: track, per class, which seeds actually contain that class in
        # y_true (n_true > 0). A class absent from y_true makes precision/
        # recall mathematically 0 for that seed regardless of model quality,
        # which can dominate the std reported across seeds.
        class_present_seeds = {c: [] for c in range(3)}
        class_absent_seeds  = {c: [] for c in range(3)}
        class_n_true        = {c: {} for c in range(3)}
        for s in seeds:
            d = prop[s]
            m = compute_metrics(d['y_true'], d['y_pred'])
            accs.append(m['accuracy'])
            y_true_arr = np.asarray(d['y_true'])
            counts = np.bincount(y_true_arr, minlength=3)
            for c in range(3):
                P[c].append(m['precision_per_class'][c])
                R[c].append(m['recall_per_class'][c])
                F[c].append(m['f1_per_class'][c])
                class_n_true[c][s] = int(counts[c])
                if counts[c] > 0:
                    class_present_seeds[c].append(s)
                else:
                    class_absent_seeds[c].append(s)
        # median-accuracy seed
        med_seed = seeds[int(np.argsort(accs)[len(accs) // 2])]
        med = compute_metrics(prop[med_seed]['y_true'], prop[med_seed]['y_pred'])

        for c in range(3):
            pm, ps = agg(P[c]); rm, rs = agg(R[c]); fm, fs = agg(F[c])
            all_rows.append({
                'speed': spd, 'class': CLASS_NAMES[c],
                'precision_mean': round(pm, 2), 'precision_std': round(ps, 2),
                'recall_mean': round(rm, 2),    'recall_std': round(rs, 2),
                'f1_mean': round(fm, 2),        'f1_std': round(fs, 2),
                'precision_medseed': round(med['precision_per_class'][c] * 100, 2),
                'recall_medseed':    round(med['recall_per_class'][c] * 100, 2),
                'f1_medseed':        round(med['f1_per_class'][c] * 100, 2),
                'median_seed': med_seed,
                # NEW fields
                'n_seeds_total': len(seeds),
                'n_seeds_with_class': len(class_present_seeds[c]),
                'seeds_missing_class': class_absent_seeds[c],
                'n_true_per_seed': class_n_true[c],
            })

    print('\n' + '=' * 100)
    print('  T7 — Per-class metrics (mean ± std across 20 seeds; % )')
    print('=' * 100)
    print(f'{"Spd":>4s}{"Class":>14s}{"Precision":>16s}{"Recall":>16s}{"F1":>16s}'
          f'{"Seeds w/ class":>18s}')
    print('-' * 100)
    for r in all_rows:
        flag = f'{r["n_seeds_with_class"]}/{r["n_seeds_total"]}'
        if r['seeds_missing_class']:
            flag += f' (missing: {",".join(map(str, r["seeds_missing_class"]))})'
        print(f'{r["speed"]:>4d}{r["class"]:>14s}'
              f'{r["precision_mean"]:>10.2f}±{r["precision_std"]:<4.1f}'
              f'{r["recall_mean"]:>10.2f}±{r["recall_std"]:<4.1f}'
              f'{r["f1_mean"]:>10.2f}±{r["f1_std"]:<4.1f}'
              f'{flag:>18s}')
        if r['n_seeds_with_class'] < r['n_seeds_total']:
            print(f'      NOTE: class absent (n_true=0) in seed(s) '
                  f'{r["seeds_missing_class"]} -> precision/recall forced to 0 '
                  f'on those seeds, inflating std.')

    with open(os.path.join(outdir, 't7_per_class.json'), 'w') as f:
        json.dump(all_rows, f, indent=2)

    # LaTeX (mean ± std version; matches review Section 6.4 template)
    has_missing = any(r['seeds_missing_class'] for r in all_rows)
    lines = [r'\begin{table}[H]',
             r'\caption{Per-class precision, recall, and F1 (mean $\pm$ std across 20 seeds).'
             + (r' Superscript $^a$ marks classes absent (zero true samples) in one or more '
                r'seeds'' test sets, which mechanically forces precision/recall to 0 on those '
                r'seeds and inflates the reported std.}' if has_missing else '}'),
             r'\centering', r'\begin{tabular}{llrrr}', r'\toprule',
             r'Speed & Class & Precision & Recall & F1 \\', r'\midrule']
    for r in all_rows:
        mark = r'$^a$' if r['seeds_missing_class'] else ''
        lines.append(f'{r["speed"]} m/s & {r["class"]}{mark} & '
                     f'{r["precision_mean"]:.2f} $\\pm$ {r["precision_std"]:.2f} & '
                     f'{r["recall_mean"]:.2f} $\\pm$ {r["recall_std"]:.2f} & '
                     f'{r["f1_mean"]:.2f} $\\pm$ {r["f1_std"]:.2f} \\\\')
    lines += [r'\bottomrule', r'\end{tabular}']
    if has_missing:
        notes = []
        for r in all_rows:
            if r['seeds_missing_class']:
                seeds_str = ', '.join(map(str, r['seeds_missing_class']))
                notes.append(f'{r["speed"]} m/s {r["class"]}: absent in seed(s) {seeds_str} '
                            f'({r["n_seeds_with_class"]}/{r["n_seeds_total"]} seeds represented)')
        lines.append(r'\par\smallskip\noindent\footnotesize $^a$' + '; '.join(notes) + '.')
    lines += [r'\end{table}']
    with open(os.path.join(outdir, 't7_table.tex'), 'w') as f:
        f.write('\n'.join(lines))
    return all_rows

In [13]:
RESULTS_DIR = '/content/drive/MyDrive/AMSTE_Results'   # sesuaikan path Anda
OUTDIR = 'outputs_posthoc'

os.makedirs(OUTDIR, exist_ok=True)
run_T4(RESULTS_DIR, OUTDIR)
run_T7(RESULTS_DIR, OUTDIR)
print(f'\nSaved JSON + LaTeX to {OUTDIR}/')


  T4 — Strengthened statistics (n=5): paired t, Wilcoxon, bootstrap CI, paired d
Comparison                 Spd   Δmean      t-p  Wilcox-p       bootCI95 (pp)      d same dir
------------------------------------------------------------------------------------------------
Proposed vs RF-Stat         25   -1.83   0.0381    0.1250      [-2.83, -0.73]  -1.36       no
Proposed vs BERT WordPiece  25   +0.45   0.7140    0.8750      [-1.48, +2.38]  +0.18       no
Proposed vs LSTM-RELU-IO-L1  25  +33.37   0.0464    0.0625    [+14.09, +54.68]  +1.27      yes
Proposed vs RF-Stat         15  +19.00   0.0079    0.0625    [+11.95, +25.59]  +2.20      yes
Proposed vs BERT WordPiece  15  +20.92   0.0131    0.0625    [+11.91, +29.54]  +1.90      yes
Proposed vs LSTM-RELU-IO-L1  15  +38.35   0.0405    0.0625    [+17.20, +62.27]  +1.33      yes

  T7 — Per-class metrics (mean ± std across 5 seeds; % )
 Spd         Class       Precision          Recall              F1    Seeds w/ class
------------------

In [14]:
run_T4(RESULTS_DIR, OUTDIR)
run_T7(RESULTS_DIR, OUTDIR)


  T4 — Strengthened statistics (n=5): paired t, Wilcoxon, bootstrap CI, paired d
Comparison                 Spd   Δmean      t-p  Wilcox-p       bootCI95 (pp)      d same dir
------------------------------------------------------------------------------------------------
Proposed vs RF-Stat         25   -1.83   0.0381    0.1250      [-2.83, -0.73]  -1.36       no
Proposed vs BERT WordPiece  25   +0.45   0.7140    0.8750      [-1.48, +2.38]  +0.18       no
Proposed vs LSTM-RELU-IO-L1  25  +33.37   0.0464    0.0625    [+14.09, +54.68]  +1.27      yes
Proposed vs RF-Stat         15  +19.00   0.0079    0.0625    [+11.95, +25.59]  +2.20      yes
Proposed vs BERT WordPiece  15  +20.92   0.0131    0.0625    [+11.91, +29.54]  +1.90      yes
Proposed vs LSTM-RELU-IO-L1  15  +38.35   0.0405    0.0625    [+17.20, +62.27]  +1.33      yes

  T7 — Per-class metrics (mean ± std across 5 seeds; % )
 Spd         Class       Precision          Recall              F1    Seeds w/ class
------------------

[{'speed': 25,
  'class': 'Interference',
  'precision_mean': np.float64(78.0),
  'precision_std': np.float64(43.82),
  'recall_mean': np.float64(78.28),
  'recall_std': np.float64(43.92),
  'f1_mean': np.float64(78.05),
  'f1_std': np.float64(43.7),
  'precision_medseed': 0.0,
  'recall_medseed': 0.0,
  'f1_medseed': 0.0,
  'median_seed': 789,
  'n_seeds_total': 5,
  'n_seeds_with_class': 4,
  'seeds_missing_class': [789],
  'n_true_per_seed': {42: 45, 123: 27, 456: 93, 789: 0, 2024: 34}},
 {'speed': 25,
  'class': 'Reactive',
  'precision_mean': np.float64(97.33),
  'precision_std': np.float64(5.96),
  'recall_mean': np.float64(99.57),
  'recall_std': np.float64(0.95),
  'f1_mean': np.float64(98.36),
  'f1_std': np.float64(3.11),
  'precision_medseed': 86.67,
  'recall_medseed': 100.0,
  'f1_medseed': 92.86,
  'median_seed': 789,
  'n_seeds_total': 5,
  'n_seeds_with_class': 5,
  'seeds_missing_class': [],
  'n_true_per_seed': {42: 69, 123: 47, 456: 118, 789: 13, 2024: 82}},
 {'speed

In [15]:
import os
print(os.listdir(OUTDIR))

['t4_statistics.json', 't4_table.tex', 't7_per_class.json', 't7_table.tex']


In [16]:
import shutil

dest = os.path.join(RESULTS_DIR, 'outputs_posthoc_FINAL')
os.makedirs(dest, exist_ok=True)

for fname in ['t4_statistics.json', 't4_table.tex', 't7_per_class.json', 't7_table.tex']:
    shutil.copy(os.path.join(OUTDIR, fname), os.path.join(dest, fname))

print(f"4 file tersimpan di: {dest}")
print(os.listdir(dest))

4 file tersimpan di: /content/drive/MyDrive/AMSTE_Results/outputs_posthoc_FINAL
['t4_statistics.json', 't4_table.tex', 't7_per_class.json', 't7_table.tex']


In [17]:
import os, datetime, glob

print("=== Timestamp file di baselines_15ms ===")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, 'baselines_15ms', '*.json'))):
    mtime = os.path.getmtime(f)
    print(os.path.basename(f), '->', datetime.datetime.fromtimestamp(mtime))

=== Timestamp file di baselines_15ms ===
bert_mitig_low_lr1e5_15ms_seed123.json -> 2026-05-22 07:52:00
bert_mitig_low_lr1e5_15ms_seed2024.json -> 2026-05-22 07:54:12
bert_mitig_low_lr1e5_15ms_seed42.json -> 2026-05-22 07:51:16
bert_mitig_low_lr1e5_15ms_seed456.json -> 2026-05-22 07:52:34
bert_mitig_low_lr1e5_15ms_seed789.json -> 2026-05-22 07:53:27
bert_mitig_warmup_lr2e5_15ms_seed123.json -> 2026-05-22 06:36:12
bert_mitig_warmup_lr2e5_15ms_seed2024.json -> 2026-05-22 08:16:41
bert_mitig_warmup_lr2e5_15ms_seed42.json -> 2026-05-22 06:02:41
bert_mitig_warmup_lr2e5_15ms_seed456.json -> 2026-05-22 07:01:49
bert_mitig_warmup_lr2e5_15ms_seed789.json -> 2026-05-22 07:42:19
bert_wp_15ms_seed123.json -> 2026-05-22 07:59:20
bert_wp_15ms_seed2024.json -> 2026-05-22 08:01:12
bert_wp_15ms_seed42.json -> 2026-05-22 07:58:40
bert_wp_15ms_seed456.json -> 2026-05-22 07:59:54
bert_wp_15ms_seed789.json -> 2026-05-22 08:00:31
cohens_d_baseline.json -> 2026-05-25 08:20:21
lstm_relu_iol1_15ms_seed123.json 

In [18]:
import json, os

SEEDS_CHECK = [42, 123, 456, 789, 2024, 7, 13, 21, 33, 55,
               77, 99, 111, 222, 333, 444, 555, 777, 888, 999]

def check_baseline(prefix, label, orig_note=""):
    print(f"\n=== {label} 15ms{orig_note} ===")
    found, missing = [], []
    for seed in SEEDS_CHECK:
        fp = f'{RESULTS_DIR}/baselines_15ms/{prefix}_15ms_seed{seed}.json'
        if os.path.exists(fp):
            d = json.load(open(fp))
            print(seed, 'accuracy=', round(d.get('accuracy', 0) * 100, 2))
            found.append(seed)
        else:
            missing.append(seed)
    if missing:
        print(f"  ⚠️  BELUM ADA file untuk {len(missing)} seed: {missing}")
    print(f"  -> {len(found)}/{len(SEEDS_CHECK)} seed tersedia untuk {label} 15ms")
    return found, missing

check_baseline('rf_vrs', 'RF-Stat', ' (cek vs original: 75.0/63.64/63.16/75.0/84.62)')
check_baseline('bert_wp', 'BERT WordPiece', ' (cek vs original: 77.89/67.14/80.69/54.55/71.56)')
check_baseline('lstm_relu_iol1', 'LSTM-RELU-IO-L1', ' (cek vs original: 75.0/72.73/73.68/12.5/30.77)')



=== RF-Stat 15ms (cek vs original: 75.0/63.64/63.16/75.0/84.62) ===
42 accuracy= 75.0
123 accuracy= 63.64
456 accuracy= 63.16
789 accuracy= 75.0
2024 accuracy= 84.62

=== BERT WordPiece 15ms (cek vs original: 77.89/67.14/80.69/54.55/71.56) ===
42 accuracy= 77.89
123 accuracy= 67.14
456 accuracy= 80.69
789 accuracy= 54.55
2024 accuracy= 71.56

=== LSTM-RELU-IO-L1 15ms (cek vs original: 75.0/72.73/73.68/12.5/30.77) ===
42 accuracy= 75.0
123 accuracy= 72.73
456 accuracy= 73.68
789 accuracy= 12.5
2024 accuracy= 30.77
